In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

import clickhouse_connect

# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns
pd.set_option("display.max_columns", None)

# Make sure wide DataFrames don't wrap
pd.set_option("display.width", None)

# Show the full content of each column (no '...')
pd.set_option("display.max_colwidth", None)

In [2]:
client = clickhouse_connect.get_client(
    host='yaujulxk39.eastus2.azure.clickhouse.cloud',      # or server IP
    port=8443,             # HTTP port (default)
    username='default',
    password='~gZjRLjjOJh1i',
    database='Competitive_Analysis'
)

In [3]:
# ---- Query ----
df_hum_hist_drg = client.query_df("""
    SELECT Distinct ADRG
    FROM Competitive_Analysis.DRGHistoricalReviewInformation
    WHERE Client = 'Hum'
""")
df_hum_hist_drg.shape

(862, 1)

In [4]:
df_hum_hist_drg['ADRG'].value_counts()

ADRG
065     1
0302    1
638     1
193     1
281     1
059     1
336     1
375     1
304     1
947     1
377     1
280     1
442     1
389     1
246     1
143     1
300     1
189     1
418     1
455     1
640     1
545     1
481     1
378     1
602     1
644     1
243     1
441     1
871     1
064     1
7104    1
259     1
0454    1
0891    1
1982    1
323     1
864     1
3142    1
5004    1
597     1
0231    1
3262    1
745     1
822     1
1432    1
1901    1
2222    1
2843    1
1832    1
3473    1
1414    1
6231    1
738     1
6632    1
1632    1
018     1
138     1
346     1
2301    1
3432    1
1412    1
917     1
698     1
154     1
974     1
987     1
683     1
180     1
593     1
922     1
178     1
242     1
603     1
071     1
919     1
853     1
166     1
270     1
453     1
329     1
674     1
177     1
854     1
286     1
551     1
723     1
100     1
660     1
239     1
467     1
315     1
840     1
386     1
025     1
737     1
070     1
054     1
543     1
175     1
207  

In [5]:
drg_desc = pd.read_excel(r"C:\Users\arunkumara\Downloads\20260323_DRG_Desp.xls",dtype='str')
drg_desc.shape

df_drg = df_hum_hist_drg.merge(drg_desc,left_on='ADRG',right_on='DRGCode',how='left')

df_drg = df_drg[~(df_drg['DRGCode'].isnull())]
df_drg.shape

(589, 4)

In [6]:
df_drg.head(10)

,ADRG,DRGCode,DRGDesc,Type
0,638,638,Diabetes with CC,Diag
1,193,193,Simple Pneumonia and Pleurisy with MCC,Diag
2,281,281,"Acute Myocardial Infarction, Discharged Alive with CC",Diag
3,059,059,Multiple Sclerosis and Cerebellar Ataxia with CC,Diag
4,336,336,Peritoneal Adhesiolysis with CC,Proc
5,375,375,Digestive Malignancy with CC,Diag
6,304,304,Hypertension with MCC,Diag
7,947,947,Signs and Symptoms with MCC,Diag
8,377,377,Gastrointestinal Hemorrhage with MCC,Diag
9,280,280,"Acute Myocardial Infarction, Discharged Alive with MCC",Diag


In [7]:
df_drg["DRG_TEXT"] = "DRG Code: " + df_drg["ADRG"] + \
                     ". Description: " + df_drg["DRGDesc"]

In [8]:
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SapBERT (for ICD codes)
sap_tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
sap_model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext").to(device)

# PubMedBERT (for text)
pub_tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
pub_model = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract").to(device)

sap_model.eval()
pub_model.eval()

KeyboardInterrupt: 

In [13]:
drg_desc.head(1)

,DRGCode,DRGDesc,Type
0,001,Heart Transplant or Implant of Heart Assist System with MCC,Pre-MDC


In [ ]:
# List all required columns
cols = [
    'PRIM_DX',
    'A_DX2','A_DX3','A_DX4','A_DX5','A_DX6','A_DX7','A_DX8','A_DX9','A_DX10',
    'A_DX11','A_DX12','A_DX13','A_DX14','A_DX15','A_DX16','A_DX17','A_DX18',
    'A_DX19','A_DX20','A_DX21','A_DX22','A_DX23','A_DX24','A_DX25',

    'A_PRIMDX_Desc',
    'A_DX2_Desc','A_DX3_Desc','A_DX4_Desc','A_DX5_Desc','A_DX6_Desc',
    'A_DX7_Desc','A_DX8_Desc','A_DX9_Desc','A_DX10_Desc','A_DX11_Desc',
    'A_DX12_Desc','A_DX13_Desc','A_DX14_Desc','A_DX15_Desc','A_DX16_Desc',
    'A_DX17_Desc','A_DX18_Desc','A_DX19_Desc','A_DX20_Desc','A_DX21_Desc',
    'A_DX22_Desc','A_DX23_Desc','A_DX24_Desc','A_DX25_Desc'
]


query = f"""
SELECT {",".join(cols)}
FROM Competitive_Analysis.DRGHistoricalReviewInformation"""
 
df_hist_icd = client.query_df(query)
df_hist_icd.shape

dx_pairs = [
    ('PRIM_DX', 'A_PRIMDX_Desc'),
    ('A_DX2', 'A_DX2_Desc'),
    ('A_DX3', 'A_DX3_Desc'),
    ('A_DX4', 'A_DX4_Desc'),
    ('A_DX5', 'A_DX5_Desc'),
    ('A_DX6', 'A_DX6_Desc'),
    ('A_DX7', 'A_DX7_Desc'),
    ('A_DX8', 'A_DX8_Desc'),
    ('A_DX9', 'A_DX9_Desc'),
    ('A_DX10', 'A_DX10_Desc'),
    ('A_DX11', 'A_DX11_Desc'),
    ('A_DX12', 'A_DX12_Desc'),
    ('A_DX13', 'A_DX13_Desc'),
    ('A_DX14', 'A_DX14_Desc'),
    ('A_DX15', 'A_DX15_Desc'),
    ('A_DX16', 'A_DX16_Desc'),
    ('A_DX17', 'A_DX17_Desc'),
    ('A_DX18', 'A_DX18_Desc'),
    ('A_DX19', 'A_DX19_Desc'),
    ('A_DX20', 'A_DX20_Desc'),
    ('A_DX21', 'A_DX21_Desc'),
    ('A_DX22', 'A_DX22_Desc'),
    ('A_DX23', 'A_DX23_Desc'),
    ('A_DX24', 'A_DX24_Desc'),
    ('A_DX25', 'A_DX25_Desc')
]

df_long = pd.concat([
    df_hist_icd[[dx, desc]].rename(columns={dx: 'DX_CODE', desc: 'DX_DESC'})
    for dx, desc in dx_pairs
], ignore_index=True)

df_long = df_long.dropna(subset=['DX_CODE'])
df_long = df_long.drop_duplicates(subset=['DX_CODE', 'DX_DESC']).reset_index(drop=True)
df_long['DX_CODE'] = (
    df_long['DX_CODE']
    .astype(str)
    .str.replace(r'[-_]?MCC', '', regex=True)
    .str.replace(r'[-_]?CC', '', regex=True)
    .str.replace('-', '', regex=False)   # remove dash
    .str.strip()
    .str.upper()
)
df_long = df_long.drop_duplicates(subset=['DX_CODE', 'DX_DESC']).reset_index(drop=True)
df_long.shape


(971009, 50)

In [23]:
px_cols = [
    'A_PX1','A_PX2','A_PX3','A_PX4','A_PX5','A_PX6','A_PX7','A_PX8','A_PX9','A_PX10',
    'A_PX11','A_PX12','A_PX13','A_PX14','A_PX15','A_PX16','A_PX17','A_PX18',
    'A_PX19','A_PX20','A_PX21','A_PX22','A_PX23','A_PX24','A_PX25',

    'A_PX1_Desc','A_PX2_Desc','A_PX3_Desc','A_PX4_Desc','A_PX5_Desc',
    'A_PX6_Desc','A_PX7_Desc','A_PX8_Desc','A_PX9_Desc','A_PX10_Desc',
    'A_PX11_Desc','A_PX12_Desc','A_PX13_Desc','A_PX14_Desc','A_PX15_Desc',
    'A_PX16_Desc','A_PX17_Desc','A_PX18_Desc','A_PX19_Desc','A_PX20_Desc',
    'A_PX21_Desc','A_PX22_Desc','A_PX23_Desc','A_PX24_Desc','A_PX25_Desc'
]

# Update query
query = f"""
SELECT {",".join(cols + px_cols)}
FROM Competitive_Analysis.DRGHistoricalReviewInformation
"""

df_hist_icd = client.query_df(query)

px_pairs = [(f'A_PX{i}', f'A_PX{i}_Desc') for i in range(1, 26)]

df_px_long = pd.concat([
    df_hist_icd[[px, desc]].rename(columns={px: 'PX_CODE', desc: 'PX_DESC'})
    for px, desc in px_pairs
], ignore_index=True)

df_px_long = df_px_long.dropna(subset=['PX_CODE'])

df_px_long['PX_CODE'] = (
    df_px_long['PX_CODE']
    .astype(str)
    .str.strip()
    .str.upper()
)

df_px_long = df_px_long.drop_duplicates(subset=['PX_CODE', 'PX_DESC']).reset_index(drop=True)

print(df_px_long.shape)
df_px_long.head()

(20360, 2)


,PX_CODE,PX_DESC
0,B2111ZZ,Fluoroscopy of Multiple Coronary Arteries using Low Osmolar Contrast
1,0DSL0ZZ,"Reposition Transverse Colon, Open Approach"
2,0DBU4ZX,"Excision of Omentum, Percutaneous Endoscopic Approach, Diagnostic"
3,5A1D70Z,"Performance of Urinary Filtration, Intermittent, Less than 6 Hours Per Day"
4,0W3P8ZZ,"Control Bleeding in Gastrointestinal Tract, Via Natural or Artificial Opening Endoscopic"


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

C:\Users\arunkumara\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arunkumara\.cache\huggingface\hub\models--medicalai--ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): 

In [27]:
df_px_long["TEXT"] = df_px_long["PX_DESC"].fillna("").astype(str)

In [29]:
def get_embeddings_batch(text_list, max_len=64):
    text_list = ["" if pd.isna(t) else str(t) for t in text_list]

    encoded = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)

    embeddings = summed / counts
    return embeddings.cpu().numpy().astype("float32")

In [30]:
batch_size = 64
texts = df_px_long["TEXT"].tolist()

all_embeddings = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    emb = get_embeddings_batch(batch)
    all_embeddings.append(emb)

all_embeddings = np.vstack(all_embeddings)

In [31]:
emb_dim = all_embeddings.shape[1]

emb_df = pd.DataFrame(
    all_embeddings,
    columns=[f"PX_EMB_{i}" for i in range(emb_dim)]
)

df_px_final = pd.concat(
    [df_px_long.reset_index(drop=True), emb_df],
    axis=1
)

sample_df = df_px_final.sample(5, random_state=42)

texts = sample_df["TEXT"].tolist()
embeddings = sample_df[[f"PX_EMB_{i}" for i in range(emb_dim)]].values

In [42]:
sample_df = df_px_final.sample(100, random_state=42)

texts = sample_df["TEXT"].tolist()
embeddings = sample_df[[f"PX_EMB_{i}" for i in range(emb_dim)]].values

In [43]:
sim_matrix = cosine_similarity(embeddings)

sim_df = pd.DataFrame(sim_matrix, index=texts, columns=texts)

#print(sim_df.round(3))

In [44]:
sim_df.shape

(100, 100)

In [45]:
sim_df.head()

,"Excision of Right Fallopian Tube, Percutaneous Approach","Excision of Ileocecal Valve, Percutaneous Approach","Transfer Back Skin, External Approach","Excision of Left Mandible, Percutaneous Approach","Dilation of Left Brachial Vein, Open Approach","Detachment at Left Foot, Complete, Open Approach","Drainage of Pelvic Cavity, Percutaneous Approach, Diagnostic","Replacement of Right Temporal Bone with Synthetic Substitute, Percutaneous Approach","Reposition Left Lower Femur, Percutaneous Approach","Removal of Internal Fixation Device from Skull, Open Approach",DILATION LT COMMON CAROTID ARTERY BIFURCATION PC,"Introduction of Other Therapeutic Substance into Lower GI, Via Natural or Artificial Opening Endoscopic","Insertion of Other Device into Ureter, Via Natural or Artificial Opening Endoscopic","Drainage of Cecum with Drainage Device, Percutaneous Approach","Revision of Synthetic Substitute in Right Patella, Open Approach","Repair Cecum, Percutaneous Approach","Drainage of Left Mandible, Percutaneous Approach, Diagnostic","Replacement of Right Foot Skin with Nonautologous Tissue Substitute, Partial Thickness, External Approach","Release Brain, Percutaneous Approach","Extirpation of Matter from Thoracolumbar Vertebral Joint, Open Approach","Supplement Left Knee Joint, Tibial Surface with Liner, Open Approach",Dressing of Right Inguinal Region using Bandage,"Excision of Left Main Bronchus, Via Natural or Artificial Opening Endoscopic","Insertion of Infusion Device into Right Internal Carotid Artery, Percutaneous Endoscopic Approach","Removal of Drainage Device from Stomach, Percutaneous Approach","Removal of Other Device from Genitourinary Tract, External Approach","Excision of Right Large Intestine, Percutaneous Endoscopic Approach, Hand-Assisted","Destruction of Left Lower Femur, Open Approach","Removal of Synthetic Substitute from Left Pleural Cavity, Open Approach","Dilation of Celiac Artery with Intraluminal Device, Open Approach","Occlusion of Portal Vein with Intraluminal Device, Percutaneous Approach","Fragmentation in Genitourinary Tract, Via Natural or Artificial Opening Endoscopic","Insertion of Artificial Sphincter into Urethra, Via Natural or Artificial Opening Endoscopic","Extirpation of Matter from Stomach, Via Natural or Artificial Opening","Excision of Right Tibia, Percutaneous Endoscopic Approach","Restriction of Left Cephalic Vein, Open Approach","Drainage of Right Axilla with Drainage Device, Open Approach","Drainage of Left Elbow Joint, Percutaneous Approach, Diagnostic","Inspection of Retroperitoneum, Open Approach","Revision of Synthetic Substitute in Right Lower Extremity, Open Approach","Dilation of Superior Mesenteric Artery with Drug-eluting Intraluminal Device, Percutaneous Approach","Destruction of Bladder, Open Approach","Insertion of Infusion Device into Left Knee Joint, Open Approach","Revision of Drainage Device in Lower Intestinal Tract, External Approach","Inspection of Left Femoral Region, Percutaneous Approach","Reposition Uterus, Open Approach","Drainage of Neck Skin, External Approach","Bypass Right Popliteal Artery to Foot Artery with Synthetic Substitute, Open Approach","Excision of Left Large Intestine, Via Natural or Artificial Opening Endoscopic, Diagnostic","Occlusion of Left Temporal Artery, Open Approach","Magnetic Resonance Imaging (MRI) of Right Upper Arm using Other Contrast, Unenhanced and Enhanced","Repair Stomach, Pylorus, Via Natural or Artificial Opening Endoscopic","Insertion of Infusion Device into Left Knee Region, Open Approach","Revision of Synthetic Substitute in Cerebral Ventricle, Open Approach","Replacement of Left Foot Skin with Autologous Tissue Substitute, Cell Suspension Technique, External Approach","Measurement of Peripheral Nervous Conductivity, Motor, External Approach","Removal of External Fixation Device from Left Toe Phalanx, External Approach",Fluoroscopy of Left Knee,"Detachment at Right 4th Toe, Low, Open Approach","Removal of Neurostimulator Lead f

In [6]:
df_hum_hist.head()

,ADRG
0,291
1,811
2,304
3,794
4,377


In [ ]:
ADRG PRIM_DX
A_DX2	A_DX3	A_DX4	A_DX5	A_DX6	A_DX7	A_DX8	A_DX9	A_DX10	A_DX11	A_DX12	A_DX13	A_DX14	A_DX15	A_DX16	A_DX17	A_DX18	A_DX19	A_DX20	A_DX21	A_DX22	A_DX23	A_DX24	A_DX25	

A_PRIMDX_Desc	A_DX2_Desc	A_DX3_Desc	A_DX4_Desc	A_DX5_Desc	A_DX6_Desc	A_DX7_Desc	A_DX8_Desc	A_DX9_Desc	A_DX10_Desc	A_DX11_Desc	A_DX12_Desc	A_DX13_Desc	A_DX14_Desc	A_DX15_Desc	A_DX16_Desc	A_DX17_Desc	A_DX18_Desc	A_DX19_Desc	A_DX20_Desc	A_DX21_Desc	A_DX22_Desc	A_DX23_Desc	A_DX24_Desc	A_DX25_Desc	
